# Amber Advanced Tutorial 3: Section 1
## Preparation of the SUSTAINER Residue

This notebook automates Section 1 of the [Amber Tutorial 3](https://ambermd.org/tutorials/advanced/tutorial3/section1.php). 
It is configured to run AmberTools commands inside an **Apptainer** container (.sif).

### Prerequisites
* Ensure **Apptainer** is installed.
* Update the `SIF_PATH` below to point to the project container file.

In [1]:
import os
import subprocess

# --- CONFIGURE THIS PATH ---
SIF_PATH = "amber_ready.sif" 

def run_apptainer(cmd):
    full_cmd = f"apptainer exec {SIF_PATH} {cmd}"
    print(f"Executing: {full_cmd}")
    
    # Use subprocess to capture errors
    result = subprocess.run(full_cmd, shell=True, capture_output=True, text=True)
    
    if result.returncode != 0:
        print("❌ COMMAND FAILED!")
        print("STDOUT:", result.stdout)
        print("STDERR:", result.stderr)
    else:
        print("✅ Success!")
        print(result.stdout)

# Verify the container exists
if not os.path.exists(SIF_PATH):
    print(f"Warning: Container not found at {SIF_PATH}. Please update the path.")

### 1. Download Input Files
Fetching the `sustainer.pdb` file from the tutorial repository.

In [ ]:
!wget -N https://ambermd.org/tutorials/advanced/tutorial3/files/sustainer.pdb

### 2. Run Antechamber
We generate the MOL2 file with AM1-BCC charges for the SUSTAINER residue.

In [2]:
# 1. Download
# !wget -N https://ambermd.org/tutorials/advanced/tutorial3/files/ras-raf.pdb

# 2. Clean and Split
with open('ras-raf.pdb', 'r') as f:
    lines = f.readlines()

# Filter out residues 243 and 244 (GTP and MG)
# Splitting into Ras (1-166) and Raf (167-242)
complex_lines = []
ras_lines = []
raf_lines = []

for line in lines:
    if line.startswith("ATOM") or line.startswith("HETATM"):
        res_num = int(line[22:26].strip())
        if res_num < 243:
            complex_lines.append(line)
            if res_num <= 166:
                ras_lines.append(line)
            else:
                raf_lines.append(line)
    elif line.startswith("TER") or line.startswith("END"):
        complex_lines.append(line)

with open('ras-raf_clean.pdb', 'w') as f: f.writelines(complex_lines)
with open('ras.pdb', 'w') as f: f.writelines(ras_lines)
with open('raf.pdb', 'w') as f: f.writelines(raf_lines)

print("Files created: ras-raf_clean.pdb, ras.pdb, raf.pdb")

Files created: ras-raf_clean.pdb, ras.pdb, raf.pdb


### 3. Tleap
Identifying missing force field parameters and generating the `frcmod` file.

In [6]:
tleap_input = """
# 1. Load the protein force field
source /opt/dat/leap/cmd/oldff/leaprc.ff99

# 2. Load the water parameters (This fixes the 'HW' error)
source /opt/dat/leap/cmd/leaprc.water.tip3p

# 3. Set the radii for MM-PBSA
set default PBRadii mbondi2

# 4. Load the cleaned structures
com = loadpdb ras-raf_clean.pdb
ras = loadpdb ras.pdb
raf = loadpdb raf.pdb

# 5. Save the vacuum parameters (Complex, Receptor, Ligand)
saveamberparm com ras-raf.prmtop ras-raf.inpcrd
saveamberparm ras ras.prmtop ras.inpcrd
saveamberparm raf raf.prmtop raf.inpcrd

# 6. Solvate the complex and save the solvated parameters
solvatebox com TIP3PBOX 12.0
saveamberparm com ras-raf_solvated.prmtop ras-raf_solvated.inpcrd

quit
"""

with open("tleap.in", "w") as f:
    f.write(tleap_input)

run_apptainer("tleap -f tleap.in")

Executing: apptainer exec amber_ready.sif tleap -f tleap.in
✅ Success!
-I: Adding /opt/dat/leap/prep to search path.
-I: Adding /opt/dat/leap/lib to search path.
-I: Adding /opt/dat/leap/parm to search path.
-I: Adding /opt/dat/leap/cmd to search path.
-f: Source tleap.in.

Welcome to LEaP!
(no leaprc in search path)
Sourcing: ./tleap.in
----- Source: /opt/dat/leap/cmd/oldff/leaprc.ff99
----- Source of /opt/dat/leap/cmd/oldff/leaprc.ff99 done
Log file: ./leap.log
Loading parameters: /opt/dat/leap/parm/parm99.dat
Reading title:
PARM99 for DNA,RNA,AA, organic molecules, Polariz.& LP incl.02/04/99
Loading library: /opt/dat/leap/lib/all_nucleic94.lib
Loading library: /opt/dat/leap/lib/all_amino94.lib
Loading library: /opt/dat/leap/lib/all_aminoct94.lib
Loading library: /opt/dat/leap/lib/all_aminont94.lib
Loading library: /opt/dat/leap/lib/ions94.lib
Loading library: /opt/dat/leap/lib/solvents.lib
----- Source: /opt/dat/leap/cmd/leaprc.water.tip3p
----- Source of /opt/dat/leap/cmd/leaprc.wa


### Create MD files


In [9]:
md_inputs = {
    "min.in": "minimise ras-raf\n &cntrl\n  imin=1,maxcyc=1000,ncyc=500,\n  cut=8.0,ntb=1,\n  ntc=2,ntf=2,\n  ntpr=100,\n  ntr=1, restraintmask=':1-242',\n  restraint_wt=2.0,\n /\n\n",
    
    "heat.in": "heat ras-raf\n &cntrl\n  imin=0,irest=0,ntx=1,\n  nstlim=25000,dt=0.002,\n  ntc=2,ntf=2,\n  cut=8.0, ntb=1,\n  ntpr=500, ntwx=500,\n  ntt=3, gamma_ln=2.0,\n  tempi=0.0, temp0=300.0, ig=-1,\n  ntr=1, restraintmask=':1-242',\n  restraint_wt=2.0,\n  nmropt=1,\n /\n &wt TYPE='TEMP0', istep1=0, istep2=25000, value1=0.1, value2=300.0, /\n &wt TYPE='END' /\n\n",

    "density.in": "density ras-raf\n &cntrl\n  imin=0,irest=1,ntx=5,\n  nstlim=25000,dt=0.002,\n  ntc=2,ntf=2,\n  cut=8.0, ntb=2, ntp=1, taup=1.0,\n  ntpr=500, ntwx=500,\n  ntt=3, gamma_ln=2.0,\n  temp0=300.0, ig=-1,\n  ntr=1, restraintmask=':1-242',\n  restraint_wt=2.0,\n /\n\n",

    "equil.in": "equil ras-raf\n &cntrl\n  imin=0,irest=1,ntx=5,\n  nstlim=250000,dt=0.002,\n  ntc=2,ntf=2,\n  cut=8.0, ntb=2, ntp=1, taup=2.0,\n  ntpr=1000, ntwx=1000,\n  ntt=3, gamma_ln=2.0,\n  temp0=300.0, ig=-1,\n /\n\n"
}

for filename, content in md_inputs.items():
    with open(filename, 'w') as f:
        f.write(content)
print("Updated MD input files with Fortran-friendly formatting.")

Updated MD input files with Fortran-friendly formatting.


### Minimization

In [10]:
# Run Minimization
# -O: Overwrite, -i: input, -o: output, -p: prmtop, -c: inpcrd, -r: restart, -ref: reference
run_apptainer("sander -O -i min.in -o min.out -p ras-raf_solvated.prmtop -c ras-raf_solvated.inpcrd -r min.rst -ref ras-raf_solvated.inpcrd")

Executing: apptainer exec amber_ready.sif sander -O -i min.in -o min.out -p ras-raf_solvated.prmtop -c ras-raf_solvated.inpcrd -r min.rst -ref ras-raf_solvated.inpcrd
✅ Success!



### 5. Verify Results
Check if the output files were created successfully.

In [ ]:
!ls -lh sus.mol2 sus.frcmod sus.lib